In [12]:
import numpy as np
import pandas as pd

df = pd.read_parquet("D:/PERSONAL/Otago/INFO501/MyChoice/ReferencePaper/Data/DHSData/PROCESSING/REPO/MYANMAR/DATA/6.MM_TEST_20SET.parquet", engine="pyarrow")

In [13]:
#normalize to numeric first
cols_to_numeric = [
    "hv024RegionDivision",
    "hv025TypeOfResidence",
    "hv219SexOfHead",
    "hv270WealthIndex",  
    "hv106_01EducationOfHead",
    "hv115_01MaritalStatus",
    "v717Occupation",
    "745aHouseOwnership",
]
for col in cols_to_numeric:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

In [14]:
def restore_ml_dtypes(df):
    df = df.copy()

    df["hv220AgeOfHead"] = df["hv220AgeOfHead"].astype("int8")
    df["hv009FamilySize"] = df["hv009FamilySize"].astype("int8")
    df["hv216HouseSize"] = df["hv216HouseSize"].astype("float64")
    df["hv014NoOfChildren"] = df["hv014NoOfChildren"].astype("int8")
    df["HouseholdClusterElevation"] = df["HouseholdClusterElevation"].astype("float64")

    df[[
        "hv024RegionDivision",
        "hv025TypeOfResidence",
        "hv219SexOfHead",
        "hv270WealthIndex",
        "hv106_01EducationOfHead",
        "hv115_01MaritalStatus",
        "v717Occupation",
        "745aHouseOwnership",
        "hv247HasBankAccount",
        "v714CurrentlyWorking",
    ]] = df[[
        "hv024RegionDivision",
        "hv025TypeOfResidence",
        "hv219SexOfHead",
        "hv270WealthIndex",
        "hv106_01EducationOfHead",
        "hv115_01MaritalStatus",
        "v717Occupation",
        "745aHouseOwnership",
        "hv247HasBankAccount",
        "v714CurrentlyWorking",
    ]].astype("category")

    df[[
        "EnergyPoor",
    ]] = df[[
        "EnergyPoor",
    ]].astype("int8")

    return df

In [15]:
df_restore = restore_ml_dtypes(df)

In [16]:
print("\nDtypes:")
print(df_restore.dtypes)

print("\nMissing values:")
print(df_restore.isna().sum().sort_values(ascending=False).head())


Dtypes:
hv270WealthIndex             category
hv024RegionDivision          category
hv025TypeOfResidence         category
hv219SexOfHead               category
hv220AgeOfHead                   int8
hv106_01EducationOfHead      category
hv115_01MaritalStatus        category
hv009FamilySize                  int8
hv247HasBankAccount          category
hv216HouseSize                float64
hv014NoOfChildren                int8
v714CurrentlyWorking         category
v717Occupation               category
745aHouseOwnership           category
HouseholdClusterElevation     float64
EnergyPoor                       int8
dtype: object

Missing values:
hv115_01MaritalStatus    2
hv270WealthIndex         0
hv024RegionDivision      0
hv025TypeOfResidence     0
hv219SexOfHead           0
dtype: int64


In [17]:
TARGET = "EnergyPoor"

drop_cols_mepi_wealth_exclusive = [

    # Per-country model
    "hv000CountryCode",

    # ID variables
    "hhidCaseIdentification",
    "hv001ClusterNumber",
    "hv002HouseholdNumber",

    # Sampling weights
    "hv005SimplilingWeight",
    "weight",

    # Wealth index (core difference)
    "hv270WealthIndex",
] 

# Build clean training dataframe
X_cols = [c for c in df_restore.columns
          if c not in drop_cols_mepi_wealth_exclusive + [TARGET]]

test_df = df_restore[X_cols + [TARGET]]

print("WEALTH-EXCLUSIVE SPECIFICATION")
print("Target:", TARGET)
print("Number of features:", len(X_cols))
print("Features used:", X_cols)

WEALTH-EXCLUSIVE SPECIFICATION
Target: EnergyPoor
Number of features: 14
Features used: ['hv024RegionDivision', 'hv025TypeOfResidence', 'hv219SexOfHead', 'hv220AgeOfHead', 'hv106_01EducationOfHead', 'hv115_01MaritalStatus', 'hv009FamilySize', 'hv247HasBankAccount', 'hv216HouseSize', 'hv014NoOfChildren', 'v714CurrentlyWorking', 'v717Occupation', '745aHouseOwnership', 'HouseholdClusterElevation']


In [18]:
print("\nDtypes:")
print(test_df.dtypes)

print("\nMissing values:")
print(test_df.isna().sum().sort_values(ascending=False).head())


Dtypes:
hv024RegionDivision          category
hv025TypeOfResidence         category
hv219SexOfHead               category
hv220AgeOfHead                   int8
hv106_01EducationOfHead      category
hv115_01MaritalStatus        category
hv009FamilySize                  int8
hv247HasBankAccount          category
hv216HouseSize                float64
hv014NoOfChildren                int8
v714CurrentlyWorking         category
v717Occupation               category
745aHouseOwnership           category
HouseholdClusterElevation     float64
EnergyPoor                       int8
dtype: object

Missing values:
hv115_01MaritalStatus    2
hv024RegionDivision      0
hv025TypeOfResidence     0
hv219SexOfHead           0
hv220AgeOfHead           0
dtype: int64


In [19]:
import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, matthews_corrcoef

# --- Paths ---
MODEL_PATH = "MYANMAR/WEALTH_EXCLUSIVE/TRADITIONAL/"
DL_MODEL_PATH = "MYANMAR/WEALTH_EXCLUSIVE/DL/"
DLFT_MODEL_PATH = "/MYANMAR/WEALTH_EXCLUSIVE/DLFT/"

# --- Load predictors ---
predictor_tr = TabularPredictor.load(MODEL_PATH)
predictor_dl = TabularPredictor.load(DL_MODEL_PATH)
predictor_ft = TabularPredictor.load(DLFT_MODEL_PATH)

LABEL = "EnergyPoor"

# -----------------------------
# chunk-safe metrics (unchanged)
# -----------------------------
def safe_binary_metrics(predictor, df, label=LABEL, chunk_size=20000, pos_label=None):
    # Decide positive label automatically if not given
    if pos_label is None:
        uniq = list(np.unique(df[label].dropna().to_numpy()))
        if 1 in uniq:
            pos_label = 1
        elif "1" in uniq:
            pos_label = "1"
        else:
            try:
                pos_label = max(uniq)
            except Exception:
                pos_label = uniq[-1]

    TP = TN = FP = FN = 0
    n = len(df)

    for start in range(0, n, chunk_size):
        chunk = df.iloc[start:start + chunk_size]
        y_true = chunk[label].to_numpy()
        y_pred = predictor.predict(chunk).to_numpy()

        yt = (y_true == pos_label)
        yp = (y_pred == pos_label)

        TP += int(np.sum(yp & yt))
        TN += int(np.sum(~yp & ~yt))
        FP += int(np.sum(yp & ~yt))
        FN += int(np.sum(~yp & yt))

    total = TP + TN + FP + FN

    accuracy = (TP + TN) / total if total else 0.0
    tpr = TP / (TP + FN) if (TP + FN) else 0.0
    tnr = TN / (TN + FP) if (TN + FP) else 0.0
    balanced_accuracy = 0.5 * (tpr + tnr)
    f1 = (2 * TP) / (2 * TP + FP + FN) if (2 * TP + FP + FN) else 0.0

    denom = (TP + FP) * (TP + FN) * (TN + FP) * (TN + FN)
    mcc = ((TP * TN - FP * FN) / np.sqrt(denom)) if denom else 0.0

    return {
        "pos_label": pos_label,
        "TP": TP, "TN": TN, "FP": FP, "FN": FN,
        "Accuracy": accuracy,
        "Balanced_Accuracy": balanced_accuracy,
        "F1": f1,
        "MCC": mcc,
    }

# --- Function for Traditional + DL (leaderboard) ---
def get_formatted_results(predictor, test_df, country_label):
    lb = predictor.leaderboard(
        data=test_df,
        extra_metrics=["f1", "mcc", "balanced_accuracy"],
        silent=True
    )

    results = (
        lb[[
            "model",
            "score_test",
            "balanced_accuracy",
            "f1",
            "mcc"
        ]]
        .rename(columns={
            "model": "Model",
            "score_test": "Accuracy",
            "balanced_accuracy": "Balanced Accuracy",
            "f1": "F1",
            "mcc": "MCC"
        })
    )

    results.insert(0, "Country", country_label)
    return results

# --- Function for FT (uses safe_binary_metrics + chunk logic) ---
def get_ft_results(
    predictor,
    test_df,
    country_label,
    chunk_threshold=50000,
    chunk_size=20000,
    pos_label=1
):
    # Get model name safely
    model_name = predictor.leaderboard(silent=True)["model"].iloc[0]

    # Large dataset -> chunk metrics
    if len(test_df) >= chunk_threshold:
        metrics = safe_binary_metrics(
            predictor,
            test_df,
            label=LABEL,
            chunk_size=chunk_size,
            pos_label=pos_label
        )
        acc = metrics["Accuracy"]
        bal_acc = metrics["Balanced_Accuracy"]
        f1 = metrics["F1"]
        mcc = metrics["MCC"]

    # Small dataset -> normal metrics
    else:
        y_true = test_df[LABEL]
        y_pred = predictor.predict(test_df, model=model_name)

        acc = accuracy_score(y_true, y_pred)
        bal_acc = balanced_accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, pos_label=pos_label)
        mcc = matthews_corrcoef(y_true, y_pred)

    results = pd.DataFrame([{
        "Country": country_label,
        "Model": model_name,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "F1": f1,
        "MCC": mcc
    }])

    return results

COUNTRY = "Myanmar_Wealth_Exclusive"


# --- Create results from 3 predictors ---
res_tr = get_formatted_results(predictor_tr, test_df, COUNTRY)
res_dl = get_formatted_results(predictor_dl, test_df, COUNTRY)
res_ft = get_ft_results(predictor_ft, test_df, COUNTRY)  # auto uses chunking only if needed

# --- Combine all (8 models total) ---
results_all = pd.concat([res_tr, res_dl, res_ft], ignore_index=True)

# --- Remove ensemble rows if any (safety) ---
results_all = results_all[
    ~results_all["Model"].str.contains("WeightedEnsemble", na=False)
]

# --- Sort by Accuracy highest first ---
results_all = results_all.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

# --- Round to 3 decimals ---
metric_cols = ["Accuracy", "Balanced Accuracy", "F1", "MCC"]
results_all[metric_cols] = results_all[metric_cols].astype(float).round(3)

results_all

,Country,Model,Accuracy,Balanced Accuracy,F1,MCC
0,Myanmar_Wealth_Exclusive,CatBoost,0.869,0.761,0.919,0.590
1,Myanmar_Wealth_Exclusive,LightGBM,0.867,0.768,0.917,0.589
2,Myanmar_Wealth_Exclusive,RandomForest,0.855,0.734,0.911,0.540
3,Myanmar_Wealth_Exclusive,NeuralNetFTTransformer,0.855,0.731,0.911,0.539
4,Myanmar_Wealth_Exclusive,NeuralNetTorch,0.849,0.735,0.906,0.526
5,Myanmar_Wealth_Exclusive,ExtraTrees,0.847,0.724,0.906,0.514
6,Myanmar_Wealth_Exclusive,NeuralNetFastAI,0.845,0.729,0.904,0.515
7,Myanmar_Wealth_Exclusive,NeuralNetTabNet,0.839,0.703,0.902,0.483


In [20]:
results_all.to_csv("MYANMAR/RESULT/TableA2_myanmar_model_performance_wealth_exclusive.csv", index=False)